In [ ]:
!pip install pyvis networkx rapidfuzz transformers torch sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 62.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 62.1 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive/', force_remount=True)

Mounted at /content/drive/


In [ ]:
import sys
import os
PROJECT_PATH = "/content/drive/MyDrive/AMS3"
sys.path.append(PROJECT_PATH)

# Character Network Extraction — Rule-Based NER (No AI Models)

**Workflow** (identical pipeline, manual NER replaces AI ensemble):
1. **Setup**: Import libraries and configure paths
2. **NER Extraction** (fast — rule-based): Extract characters from all chapters → store in memory
3. **Co-occurrence + Submission** (fast, re-run as needed): Use cached NER results with different parameters

**NER strategies used** (see `nlp_manual_ner.py`):
1. Gazetteer lookup (characters.txt + hard-coded Asimov names)
2. Capitalization heuristics (unknown proper nouns mid-sentence)
3. Regex patterns (titled names: Docteur X, R. Name, etc.)

## 1. Setup & Imports

In [ ]:
import importlib
import networkx as nx
import pandas as pd
import time
from pathlib import Path
from collections import Counter
import re

# Reload modules
importlib.reload(importlib.import_module("nlp_manual_ner"))
importlib.reload(importlib.import_module("nlp_cooccurrence"))
importlib.reload(importlib.import_module("nlp_extract_characters"))
importlib.reload(importlib.import_module("nlp_aliases"))
importlib.reload(importlib.import_module("nlp_graph"))
importlib.reload(importlib.import_module("nlp_relation"))
importlib.reload(importlib.import_module("nlp_visualize_web"))

from nlp_visualize_web import create_combined_html
from nlp_manual_ner import manual_extract_entities, precompute_dynamic_blocklist, ASIMOV_CHARACTERS, ASIMOV_LOCATIONS
from nlp_extract_characters import count_entities, filter_persons, filter_locations
from nlp_utils import read_file, load_anti_dict
from nlp_aliases import group_aliases, alias_dictionary, merge_alias_counts, filter_by_frequency, apply_manual_aliases, apply_gazetteer_aliases
from nlp_cooccurrence import detect_cooccurrences
from nlp_graph import generate_graph, remove_isolated_nodes
from nlp_relation import label_relationships, print_validation_report

print("✅ Modules loaded (rule-based NER — no AI models!)")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-xlm-roberta-base-sentiment
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Modules loaded (rule-based NER — no AI models!)


## 2. Configuration

In [ ]:
# Data paths
books_folder = "/content/drive/MyDrive/AMS3/data"
books_config = [
    (list(range(0, 19)), "paf", f"{books_folder}/prelude_a_fondation"),
    (list(range(0, 18)), "lca", f"{books_folder}/les_cavernes_d_acier"),
]

# Anti-dictionary for filtering false positives
anti_dict_file = "/content/drive/MyDrive/AMS3/antidict.txt"
anti_dict = load_anti_dict(anti_dict_file)

# ======================================================================
# CUSTOM GAZETTEER (optional — set to None to use default Asimov gazetteers)
# ======================================================================
# To use with a different book, replace these with your own dictionaries:
#   custom_chars = {"Frodo Baggins": ["Frodo"], "Gandalf": ["Mithrandir"]}
#   custom_locs  = {"Mordor": ["Mordor"], "The Shire": ["Shire"]}
# Then set:
#   GAZETTEER_CHARACTERS = custom_chars
#   GAZETTEER_LOCATIONS = custom_locs
#   TITLE_PREFIXES = r"Lord|Lady|Sir|Mr\.?|Mrs\.?|Dr\.?|Captain|King|Queen"
#   ENABLE_ROBOT_PATTERN = False
# ======================================================================
GAZETTEER_CHARACTERS = None   # None → uses ASIMOV_CHARACTERS
GAZETTEER_LOCATIONS  = None   # None → uses ASIMOV_LOCATIONS
TITLE_PREFIXES       = None   # None → uses default French titles
ENABLE_ROBOT_PATTERN = True   # Set False for non-Asimov texts

# Resolve the actual gazetteers for display
_char_gaz = GAZETTEER_CHARACTERS if GAZETTEER_CHARACTERS is not None else ASIMOV_CHARACTERS
_loc_gaz  = GAZETTEER_LOCATIONS if GAZETTEER_LOCATIONS is not None else ASIMOV_LOCATIONS
_gaz_label = "custom" if GAZETTEER_CHARACTERS is not None else "Asimov (default)"

print(f"📚 Books configured: {len(books_config)} books")
print(f"🚫 Anti-dictionary loaded: {len(anti_dict)} entries")
print(f"📖 Gazetteer: {_gaz_label} ({len(_char_gaz)} characters, {len(_loc_gaz)} locations)")

📚 Books configured: 2 books
🚫 Anti-dictionary loaded: 1012 entries


## 3. NER Extraction (Run Once - Slow ~2-5 min)

Extract all characters from all chapters and store in memory. **Only re-run this if you change NER/filtering logic.**

In [ ]:
ner_cache = {}  # {chapter_id: {'text': str, 'LP': Counter, 'LL': Counter}}

In [ ]:
# Dynamic blocklist built once from the full corpus.

all_texts = []
for chapter_range, _book_code, folder_path in books_config:
    for chapter_num in chapter_range:
        chapter_file = f"{folder_path}/chapter_{chapter_num + 1}.txt.preprocessed"
        all_texts.append(read_file(chapter_file))

dynamic_blocklist = precompute_dynamic_blocklist(
    all_texts,
    gazetteer_characters=GAZETTEER_CHARACTERS,
    gazetteer_locations=GAZETTEER_LOCATIONS,
)
print(f"Dynamic blocklist size: {len(dynamic_blocklist)}")

cap_counts = Counter()
lower_counts = Counter()
word_pattern = re.compile(r"\b[A-Za-zÀ-ÿ]+\b")

for text in all_texts:
    for match in word_pattern.finditer(text):
        word = match.group()
        if word.istitle():
            cap_counts[word.lower()] += 1
        elif word.islower():
            lower_counts[word.lower()] += 1

known_names = set()
gazetteer_forms = []
for canonical, aliases in _char_gaz.items():
    gazetteer_forms.append(canonical)
    gazetteer_forms.extend(aliases)
for canonical, aliases in _loc_gaz.items():
    gazetteer_forms.append(canonical)
    gazetteer_forms.extend(aliases)
for form in gazetteer_forms:
    for token in re.findall(r"\b[A-Za-zÀ-ÿ]+\b", form.lower()):
        known_names.add(token)

ratio_rows = []
for word in dynamic_blocklist:
    if word in known_names:
        continue
    total = cap_counts[word] + lower_counts[word]
    if total == 0:
        continue
    ratio_rows.append((word, cap_counts[word] / total, total, cap_counts[word], lower_counts[word]))

ratio_rows.sort(key=lambda x: (-x[1], -x[2], x[0]))
print("Top 20 dynamic blocklist words (word, cap_ratio, total, cap_count, lower_count):")
for word, ratio, total, cap_c, low_c in ratio_rows[:20]:
    print(f"  {word:20s} ratio={ratio:.3f} total={total:4d} cap={cap_c:4d} lower={low_c:4d}")

Dynamic blocklist size: 52
Top 20 dynamic blocklist words (word, cap_ratio, total, cap_count, lower_count):
  terriens             ratio=1.000 total=  34 cap=  34 lower=   0
  galaxie              ratio=1.000 total=  32 cap=  32 lower=   0
  ca                   ratio=1.000 total=  21 cap=  21 lower=   0
  maîtresse            ratio=1.000 total=  21 cap=  21 lower=   0
  toilettes            ratio=1.000 total=  17 cap=  17 lower=   0
  exos                 ratio=1.000 total=  14 cap=  14 lower=   0
  benastra             ratio=1.000 total=  11 cap=  11 lower=   0
  fe                   ratio=1.000 total=  11 cap=  11 lower=   0
  randa                ratio=1.000 total=  11 cap=  11 lower=   0
  dieu                 ratio=1.000 total=   7 cap=   7 lower=   0
  e                    ratio=1.000 total=   7 cap=   7 lower=   0
  marbie               ratio=1.000 total=   7 cap=   7 lower=   0
  bible                ratio=1.000 total=   6 cap=   6 lower=   0
  jenarr               ratio=1.000

In [ ]:
# Storage for NER results (in memory)

print("="*60)
print("🧠 EXTRACTING CHARACTERS (RULE-BASED NER — NO MODELS)")
print("="*60)
print()

start_time = time.time()

for chapter_range, book_code, folder_path in books_config:
    for chapter_num in chapter_range:
        chapter_id = f"{book_code}{chapter_num}"
        chapter_file = f"{folder_path}/chapter_{chapter_num + 1}.txt.preprocessed"

        print(f"Processing {chapter_id}...", end=" ", flush=True)

        # Read text
        text = read_file(chapter_file)

        # Extract entities using rule-based NER (no AI models)
        # include_locations=True so locations are tagged as LOC and excluded from persons
        raw_entities = manual_extract_entities(
            text,
            anti_dict=anti_dict,
            include_locations=True,
            dynamic_blocklist=dynamic_blocklist,
            gazetteer_characters=GAZETTEER_CHARACTERS,
            gazetteer_locations=GAZETTEER_LOCATIONS,
            title_prefixes=TITLE_PREFIXES,
            enable_robot_pattern=ENABLE_ROBOT_PATTERN,
            )

        # Count entities
        L = count_entities(raw_entities)

        # Filter persons
        LP = filter_persons(L, anti_dict=anti_dict)

        # Filter locations (optional, can be used for graph context)
        LL = filter_locations(L)

        # Store in cache (no alias grouping yet)
        ner_cache[chapter_id] = {
            'text': text,
            'L': L,
            'LP': LP,  # Raw person counts (before alias merging)
            'LL': LL
        }

        print(f"✓ {len(LP)} raw entities")

elapsed = time.time() - start_time
print()
print("="*60)
print(f"✅ NER EXTRACTION COMPLETE (rule-based)")
print(f"⏱️  Time: {elapsed:.2f}s ({elapsed/60:.2f} min)")
print(f"📊 Cached: {len(ner_cache)} chapters")
print("="*60)

🧠 EXTRACTING CHARACTERS (RULE-BASED NER — NO MODELS)

Processing paf0... ✓ 8 raw entities
Processing paf1... ✓ 6 raw entities
Processing paf2... ✓ 5 raw entities
Processing paf3... ✓ 12 raw entities
Processing paf4... ✓ 9 raw entities
Processing paf5... ✓ 8 raw entities
Processing paf6... ✓ 10 raw entities
Processing paf7... ✓ 11 raw entities
Processing paf8... ✓ 7 raw entities
Processing paf9... ✓ 5 raw entities
Processing paf10... ✓ 7 raw entities
Processing paf11... ✓ 11 raw entities
Processing paf12... ✓ 17 raw entities
Processing paf13... ✓ 15 raw entities
Processing paf14... ✓ 13 raw entities
Processing paf15... ✓ 16 raw entities
Processing paf16... ✓ 17 raw entities
Processing paf17... ✓ 14 raw entities
Processing paf18... ✓ 16 raw entities
Processing lca0... ✓ 10 raw entities
Processing lca1... ✓ 10 raw entities
Processing lca2... ✓ 5 raw entities
Processing lca3... ✓ 15 raw entities
Processing lca4... ✓ 12 raw entities
Processing lca5... ✓ 9 raw entities
Processing lca6... ✓ 1

## 3.5. Save/Load NER Cache (Optional - Persist Results)

**Save** NER results to disk so they survive kernel restarts. **Load** to skip NER extraction entirely.

In [ ]:
import pickle

# Cache file path - save in project directory (Google Drive)
project_dir = Path("/content/drive/MyDrive/AMS3")
cache_file = project_dir / "ner_cache3.pkl"

# =============================================================================
# SAVE NER CACHE TO FILE
# =============================================================================
def save_ner_cache():
    """Save the ner_cache dictionary to disk (Google Drive)"""
    if not ner_cache:
        print("⚠️  NER cache is empty. Run cell 3 first.")
        return

    with open(cache_file, 'wb') as f:
        pickle.dump(ner_cache, f)

    file_size = cache_file.stat().st_size / (1024 * 1024)  # MB
    print(f"✅ Saved {len(ner_cache)} chapters to Google Drive")
    print(f"   Location: {cache_file.name}")
    print(f"   File size: {file_size:.2f} MB")

# =============================================================================
# LOAD NER CACHE FROM FILE
# =============================================================================
def load_ner_cache():
    """Load the ner_cache dictionary from disk (Google Drive)"""
    global ner_cache

    if not cache_file.exists():
        print(f"❌ Cache file not found in Google Drive")
        print(f"   Expected: {cache_file.name}")
        print("   Run cell 3 to extract NER, then save with save_ner_cache()")
        return False

    with open(cache_file, 'rb') as f:
        ner_cache = pickle.load(f)

    file_size = cache_file.stat().st_size / (1024 * 1024)  # MB
    print(f"✅ Loaded {len(ner_cache)} chapters from Google Drive")
    print(f"   Location: {cache_file.name}")
    print(f"   File size: {file_size:.2f} MB")
    print(f"   You can now skip cell 3 and go directly to cell 4!")
    return True

# =============================================================================
# AUTO-SAVE AFTER NER EXTRACTION
# =============================================================================
def auto_save_after_ner():
    """Automatically save cache after running NER extraction"""
    if ner_cache and len(ner_cache) > 0:
        save_ner_cache()
        print("   💾 Auto-saved to Google Drive")

# =============================================================================
# USAGE
# =============================================================================
print("📁 NER Cache Management (Google Drive)")
print("-" * 60)
print(f"Cache location: {cache_file}")
print()
print("Commands available:")
print("  • save_ner_cache()       - Save cache to Google Drive")
print("  • load_ner_cache()       - Load cache from Google Drive")
print("  • auto_save_after_ner()  - Auto-save after cell 3")
print()
print("Workflow:")
print("  1. Run cell 3 (NER extraction)")
print("  2. Cache is automatically in Google Drive sync folder")
print("  3. Next session: Run load_ner_cache() to skip cell 3")
print("-" * 60)

# Auto-load if cache exists and ner_cache is empty
if cache_file.exists() and not ner_cache:
    print()
    print("🔍 Found existing cache in Google Drive!")
    print("   Run: load_ner_cache()")
elif ner_cache:
    print()
    print(f"✓ NER cache in memory: {len(ner_cache)} chapters")
    print("  To save to Google Drive: run save_ner_cache()")

📁 NER Cache Management (Google Drive)
------------------------------------------------------------
Cache location: /content/drive/MyDrive/AMS3/ner_cache3.pkl

Commands available:
  • save_ner_cache()       - Save cache to Google Drive
  • load_ner_cache()       - Load cache from Google Drive
  • auto_save_after_ner()  - Auto-save after cell 3

Workflow:
  1. Run cell 3 (NER extraction)
  2. Cache is automatically in Google Drive sync folder
  3. Next session: Run load_ner_cache() to skip cell 3
------------------------------------------------------------

✓ NER cache in memory: 37 chapters
  To save to Google Drive: run save_ner_cache()


In [ ]:
save_ner_cache()

✅ Saved 37 chapters to Google Drive
   Location: ner_cache3.pkl
   File size: 0.73 MB


In [ ]:
load_ner_cache()

✅ Loaded 37 chapters from Google Drive
   Location: ner_cache3.pkl
   File size: 0.73 MB
   You can now skip cell 3 and go directly to cell 4!


True

## 4. Co-occurrence Detection + Submission (Fast - Re-run as needed)

**Change parameters below and re-run this cell** to test different settings without re-running NER.

- `distance_max` — co-occurrence window size in words
- `min_occurrences` — minimum number of mentions to keep a character (applied after alias merging)

In [ ]:
def read_file(path):
    with open(path, "r", encoding="utf8") as f:
        return f.read()

def load_anti_dict(path="antidict.txt"):
    s = set()
    try:
        with open(path, "r", encoding="utf8") as f:
            for line in f:
                # remove inline comments and surrounding whitespace
                content = line.split("#", 1)[0].strip()
                if not content:
                    continue
                s.add(content.lower())
        return s
    except FileNotFoundError:
        return set()

## 4.5. Relation Labeling Cache

Load or initialize the relation cache before running the main loop.
The cache stores `(chapter_id, charA, charB) → edge_type` so the NLI model
is never called twice for the same pair.

**Changing `distance_max` does NOT require clearing this cache.**
Only clear it if you change `NLI_MODEL` or `CONFIDENCE_THRESHOLD` in `nlp_relation.py`.


In [ ]:
import pickle
import os

RELATION_CACHE_FILE = "relation_cache.pkl"
relation_cache = {}

if os.path.exists(RELATION_CACHE_FILE):
    with open(RELATION_CACHE_FILE, "rb") as f:
        relation_cache = pickle.load(f)
    print(f"✅ Loaded {len(relation_cache)} cached relation labels from '{RELATION_CACHE_FILE}'")
else:
    print(f"ℹ️  No relation cache found — labels will be computed from scratch.")
    print(f"   The NLI model (~350 MB) will be downloaded on first use.")


ℹ️  No relation cache found — labels will be computed from scratch.
   The NLI model (~350 MB) will be downloaded on first use.


In [ ]:
# ============================================================
# ADJUST THESE PARAMETERS AND RE-RUN THIS CELL
# ============================================================
distance_max    = 150   # Co-occurrence window size in words
min_occurrences = 2     # Minimum mentions to keep a character (after alias merging)
# ============================================================

import html
import os
from collections import Counter as _Counter

output_csv = f"submission_{distance_max}_min{min_occurrences}.csv"

books_folder = "/content/drive/MyDrive/AMS3/data"
books_config = [
    (list(range(0, 19)), "paf", f"{books_folder}/prelude_a_fondation"),
    (list(range(0, 18)), "lca", f"{books_folder}/les_cavernes_d_acier"),
]

# ============================================================
# MANUAL ALIAS OVERRIDES — add/comment out entries as needed
# ============================================================
MANUAL_ALIASES = {
    "Empereur":   "Cléon",
    "l'Empereur": "Cléon",
    "L’Empereur": "Cléon",
    "Sire":       "Cléon",
    "Cléon Ier":  "Cléon",
}
# ============================================================

print("="*60)
print(f"🔗 DETECTING CO-OCCURRENCES + LABELING RELATIONSHIPS")
print(f"   window={distance_max}, min_occ={min_occurrences}")
print("="*60)
print()

if not ner_cache:
    print("❌ ERROR: NER cache is empty! Run the NER extraction cell first.")
else:
    start_time = time.time()

    submission_data = []

    # Store per-chapter data for the validation report (section 5)
    edge_labels_per_chapter   = {}
    alias_map_per_chapter     = {}

    # Tally of all edge_type labels across the full run
    global_label_counts = _Counter()

    for chapters, book_code, folder_path in books_config:
        print(f"\n📖 Processing book: {book_code} ({len(chapters)} chapters)")
        for chapter_num in chapters:
            chapter_id = f"{book_code}{chapter_num}"

            chapter_file = os.path.join(
                folder_path,
                f"chapter_{chapter_num + 1}.txt.preprocessed"
            )

            text = read_file(chapter_file)

            if not os.path.isfile(chapter_file):
                print(f"   ⚠️  {chapter_id}: File not found: {chapter_file}")
                continue

            try:
                print(f"   📄 Processing {chapter_id}...", end=" ", flush=True)

                LP = ner_cache[chapter_id]['LP']

                groups    = group_aliases(LP)
                alias_map = alias_dictionary(groups)
                alias_map = apply_gazetteer_aliases(alias_map, ASIMOV_CHARACTERS, LP)
                alias_map = apply_manual_aliases(alias_map, MANUAL_ALIASES)
                LP_merged = merge_alias_counts(LP, alias_map)

                LP_before = len(LP_merged)
                LP_merged = filter_by_frequency(LP_merged, min_occurrences)
                LP_after  = len(LP_merged)

                cooccurrences = detect_cooccurrences(text, LP_merged, distance_max)

                # # ── Relationship labeling (new) ─────────────────────────────
                # edge_labels = label_relationships(
                #     text,
                #     cooccurrences,
                #     alias_map,
                #     distance_max=distance_max,
                #     chapter_id=chapter_id,
                #     cache=relation_cache,
                # )

                # Store for validation report
                # edge_labels_per_chapter[chapter_id] = edge_labels
                alias_map_per_chapter[chapter_id]   = alias_map

                # Tally labels
                # for lbl in edge_labels.values():
                #     global_label_counts[lbl] += 1
                # ────────────────────────────────────────────────────────────

                G = generate_graph(
                    cooccurrences,
                    LP_merged,
                    alias_map=alias_map,
                    # edge_labels=edge_labels,
                )

                nodes_before = G.number_of_nodes()
                remove_isolated_nodes(G)
                nodes_removed = nodes_before - G.number_of_nodes()

                graphml_str = "".join(nx.generate_graphml(G))
                graphml_str = html.unescape(graphml_str)

                submission_data.append({'ID': chapter_id, 'graphml': graphml_str})

                filtered_msg  = f", filtered {LP_before}→{LP_after}" if LP_before != LP_after else ""
                isolated_msg  = f", -{nodes_removed} isolated" if nodes_removed > 0 else ""
                print(f"✓ (N:{G.number_of_nodes()} E:{G.number_of_edges()}{filtered_msg}{isolated_msg})")

            except Exception as e:
                print(f"❌ Error: {e}")
                continue

    # Save submission CSV
    df_submission = pd.DataFrame(submission_data)
    df_submission.set_index('ID', inplace=True)
    df_submission.to_csv(output_csv, encoding='utf-8')

    elapsed = time.time() - start_time
    print()
    print("="*60)
    print(f"✅ SUBMISSION GENERATED")
    print(f"⏱️  Time: {elapsed:.2f}s")
    print(f"📄 File: {output_csv}")
    print(f"📊 Entries: {len(df_submission)}")
    print(f"🏷️  Edge labels: {dict(global_label_counts)}")
    print("="*60)


🔗 DETECTING CO-OCCURRENCES + LABELING RELATIONSHIPS
   window=150, min_occ=2


📖 Processing book: paf (19 chapters)
   📄 Processing paf0... ✓ (N:4 E:6, filtered 5→4)
   📄 Processing paf1... ✓ (N:4 E:6, filtered 5→4)
   📄 Processing paf2... ✓ (N:3 E:3)
   📄 Processing paf3... ✓ (N:5 E:10)
   📄 Processing paf4... ✓ (N:6 E:11)
   📄 Processing paf5... ✓ (N:4 E:6)
   📄 Processing paf6... ✓ (N:5 E:10, filtered 6→5)
   📄 Processing paf7... ✓ (N:5 E:8, filtered 6→5)
   📄 Processing paf8... ✓ (N:3 E:3, filtered 4→3)
   📄 Processing paf9... ✓ (N:2 E:1)
   📄 Processing paf10... ✓ (N:4 E:6)
   📄 Processing paf11... ✓ (N:5 E:7, filtered 6→5)
   📄 Processing paf12... ✓ (N:9 E:18)
   📄 Processing paf13... ✓ (N:8 E:18, filtered 10→8)
   📄 Processing paf14... ✓ (N:6 E:13, filtered 9→6)
   📄 Processing paf15... ✓ (N:9 E:30, filtered 10→9)
   📄 Processing paf16... ✓ (N:8 E:22, filtered 10→8)
   📄 Processing paf17... ✓ (N:7 E:21, filtered 10→7)
   📄 Processing paf18... ✓ (N:7 E:14, filtered 12→7)

📖 Proce

In [ ]:
# Save the relation cache so the NLI model is never called again for already-seen pairs.
# Run this after every successful main loop execution.
with open(RELATION_CACHE_FILE, "wb") as f:
    pickle.dump(relation_cache, f)

cache_size_kb = os.path.getsize(RELATION_CACHE_FILE) / 1024
print(f"💾 Relation cache saved → '{RELATION_CACHE_FILE}'")
print(f"   Entries : {len(relation_cache)}")
print(f"   Size    : {cache_size_kb:.1f} KB")


💾 Relation cache saved → 'relation_cache.pkl'
   Entries : 527
   Size    : 9.7 KB


## 5. Verification (Optional)

In [ ]:
# Quick verification of submission
print("📊 Submission Overview:")
print(df_submission.head(10))
print()

# Sample graph inspection
sample_id = "paf0"
if sample_id in df_submission.index:
    import io
    graphml_str = df_submission.loc[sample_id, "graphml"]
    G_sample = nx.read_graphml(io.StringIO(graphml_str))

    print(f"Sample: {sample_id}")
    print(f"  Nodes: {G_sample.number_of_nodes()}")
    print(f"  Edges: {G_sample.number_of_edges()}")
    print(f"\n  Top 5 nodes:")
    for i, (node, data) in enumerate(list(G_sample.nodes(data=True))[:5]):
        names = data.get('names', '⚠️ MISSING')
        print(f"    • {node}: {names}")

📊 Submission Overview:
                                                graphml
ID                                                     
paf0  <graphml xmlns="http://graphml.graphdrawing.or...
paf1  <graphml xmlns="http://graphml.graphdrawing.or...
paf2  <graphml xmlns="http://graphml.graphdrawing.or...
paf3  <graphml xmlns="http://graphml.graphdrawing.or...
paf4  <graphml xmlns="http://graphml.graphdrawing.or...
paf5  <graphml xmlns="http://graphml.graphdrawing.or...
paf6  <graphml xmlns="http://graphml.graphdrawing.or...
paf7  <graphml xmlns="http://graphml.graphdrawing.or...
paf8  <graphml xmlns="http://graphml.graphdrawing.or...
paf9  <graphml xmlns="http://graphml.graphdrawing.or...

Sample: paf0
  Nodes: 4
  Edges: 6

  Top 5 nodes:
    • Demerzel: Demerzel;Eto Demerzel
    • Seldon: Seldon;Hari Seldon
    • Hummin: Hummin
    • Cléon: Cléon;Sire;Empereur;l'Empereur;L’Empereur;Cléon Ier


In [ ]:
# ── Relation label spot-check ─────────────────────────────────────────────────
# Change inspect_id to any chapter you want to examine.
# Prints the most hostile and most friendly edges with their driving text snippet.
inspect_id = "paf0"

if inspect_id in edge_labels_per_chapter:
    print_validation_report(
        text         = ner_cache[inspect_id]["text"],
        edge_labels  = edge_labels_per_chapter[inspect_id],
        alias_map    = alias_map_per_chapter[inspect_id],
        distance_max = distance_max,
        chapter_id   = inspect_id,
        n            = 5,
    )
else:
    print(f"⚠️  '{inspect_id}' not found in edge_labels_per_chapter. Run the main loop first.")



  VALIDATION REPORT — chapter paf0
  Total edges: 6  |  friendly: 1  hostile: 5  neutral: 0

── HOSTILE ──
  Cléon ↔ Demerzel
  « Cléon était empereur depuis dix ans à peine et, quand le protocole l’exigeait, il y avait des moments où, pourvu qu’il fût revêtu des atours et ornements idoines, il réussissait à paraître majestueux.… »

  Cléon ↔ Seldon
  « CLÉON Ier— ... dernier Empereur galactique de la dynastie Entun. Né en l’an 11988 de l’Ère Galactique, la même année que Hari Seldon. (On pense que la date de naissance de Seldon, que certains estimen… »

  Demerzel ↔ Seldon
  « Hari Seldon ? »      Cléon était empereur depuis dix ans à peine et, quand le protocole l’exigeait, il y avait des moments où, pourvu qu’il fût revêtu des atours et ornements idoines, il réussissait à… »

  Hummin ↔ Seldon
  « Seldon fixa l’homme sans se gêner, car celui-ci semblait engagé dans quelque débat intérieur. Un instant, il donna l’impression d’être sur le point de parler, puis il parut se raviser, pu

## 7. Interactive HTML Visualization (with Relationship Labels)

Edges are **colour-coded** by relationship type:
- 🟢 **Green** → friendly
- 🔴 **Red** → hostile
- ⚫ **Grey** → neutral

Hover over an edge to see its weight and label. Change `chapters_to_visualize` to any chapter IDs you want.

In [ ]:
import io
from collections import Counter

print("\n" + "="*60)
print("COMBINED HTML VISUALIZATION (all chapters, one file)")
print("="*60)
print("Edge colours:  🟢 friendly  🔴 hostile  ⚫ neutral")
print()

# ── Which chapters to include ─────────────────────────────────────────────────
# Default: all chapters from both books.
# Narrow it down by editing this list, e.g. ["paf0", "paf1", "lca0"].
all_chapter_ids = list(df_submission.index)  # every chapter in submission order
# ─────────────────────────────────────────────────────────────────────────────

chapters_data = []

for chapter_id in all_chapter_ids:
    if chapter_id not in df_submission.index:
        print(f"⚠️  '{chapter_id}' not found in submission — skipping.")
        continue

    # Parse GraphML back to NetworkX (edge_type attributes are preserved)
    graphml_str = df_submission.loc[chapter_id, "graphml"]
    G_vis = nx.read_graphml(io.StringIO(graphml_str))

    # Reconstruct co-occurrence Counter from stored edge weights
    cooccurrences_vis = Counter()
    for u, v, data in G_vis.edges(data=True):
        weight = int(float(data.get("weight", 1)))
        cooccurrences_vis[(u, v)] = weight

    # Optional per-chapter label stats (passed as 4th element)
    label_counts = None
    if chapter_id in edge_labels_per_chapter:
        label_counts = Counter(edge_labels_per_chapter[chapter_id].values())

    chapters_data.append((chapter_id, G_vis, cooccurrences_vis, label_counts))

    n, e = G_vis.number_of_nodes(), G_vis.number_of_edges()
    lbl_str = ""
    if label_counts:
        lbl_str = (f"  friendly={label_counts.get('friendly',0)}"
                   f"  hostile={label_counts.get('hostile',0)}"
                   f"  neutral={label_counts.get('neutral',0)}")
    print(f"  ✓ {chapter_id}: {n} nodes, {e} edges{lbl_str}")

print()
output_html = f"all_networks_{distance_max}_min{min_occurrences}.html"
abs_path = create_combined_html(
    chapters_data,
    output_file=output_html,
    title=f"Character Networks  (window={distance_max}, min={min_occurrences})",
)

print()
print("="*60)
print(f"✅ Done!  {len(chapters_data)} chapters written to:")
print(f"   {abs_path}")
print("   Open the file in a browser — use the left sidebar to navigate.")
print("="*60)



COMBINED HTML VISUALIZATION (all chapters, one file)
Edge colours:  🟢 friendly  🔴 hostile  ⚫ neutral

  ✓ paf0: 4 nodes, 6 edges  friendly=1  hostile=5  neutral=0
  ✓ paf1: 5 nodes, 9 edges  friendly=1  hostile=7  neutral=1
  ✓ paf2: 3 nodes, 3 edges  friendly=0  hostile=2  neutral=1
  ✓ paf3: 5 nodes, 10 edges  friendly=3  hostile=5  neutral=2
  ✓ paf4: 6 nodes, 12 edges  friendly=3  hostile=5  neutral=4
  ✓ paf5: 6 nodes, 15 edges  friendly=0  hostile=14  neutral=1
  ✓ paf6: 6 nodes, 15 edges  friendly=1  hostile=13  neutral=1
  ✓ paf7: 5 nodes, 8 edges  friendly=2  hostile=5  neutral=1
  ✓ paf8: 3 nodes, 3 edges  friendly=3  hostile=0  neutral=0
  ✓ paf9: 2 nodes, 1 edges  friendly=1  hostile=0  neutral=0
  ✓ paf10: 4 nodes, 6 edges  friendly=1  hostile=4  neutral=1
  ✓ paf11: 5 nodes, 7 edges  friendly=1  hostile=3  neutral=3
  ✓ paf12: 9 nodes, 18 edges  friendly=7  hostile=8  neutral=3
  ✓ paf13: 8 nodes, 18 edges  friendly=6  hostile=11  neutral=1
  ✓ paf14: 6 nodes, 13 edges  